In [ ]:
"""
AlexNet is a deep convolutional neural network used for image classification. It consists of multiple convolutional and fully connected layers designed to extract
features and perform classification efficiently.

Image
  ↓
Conv
  ↓
ReLU
  ↓
Pooling
  ↓
Conv
  ↓
ReLU
  ↓
Pooling
  ↓
Conv
  ↓
ReLU
  ↓
Conv
  ↓
ReLU
  ↓
Conv
  ↓
ReLU
  ↓
Pooling
  ↓
Fully Connected
  ↓
Fully Connected
  ↓
Output
"""

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

In [3]:
# Loading and Preprocessing CIFAR-10 Dataset

# if have not downloaded dataset uncomment and use this
# Load CIFAR-10 data
# (x_train, y_train), (x_test, y_test) = cifar10.load_data()
# Normalize pixel values
# x_train = x_train.astype('float32') / 255.0
# x_test = x_test.astype('float32') / 255.0
# One-hot encode the labels
# y_train = to_categorical(y_train, 10)
# y_test = to_categorical(y_test, 10)

import pickle
import numpy as np
import os

def load_cifar10_batch(file_path):
    with open(file_path, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    data = batch[b'data']
    labels = batch[b'labels']
    data = data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return data, np.array(labels)

def load_cifar10(root='./data'):
    base_path = os.path.join(root, 'cifar-10-batches-py')
    
    x_train_list = []
    y_train_list = []
    for i in range(1, 6):
        file_path = os.path.join(base_path, f'data_batch_{i}')
        data, labels = load_cifar10_batch(file_path)
        x_train_list.append(data)
        y_train_list.append(labels)
    
    x_train = np.concatenate(x_train_list)
    y_train = np.concatenate(y_train_list)
    
    test_file = os.path.join(base_path, 'test_batch')
    x_test, y_test = load_cifar10_batch(test_file)
    
    return (x_train, y_train), (x_test, y_test)

(x_train, y_train), (x_test, y_test) = load_cifar10(root='./data')

x_train, x_test = x_train / 255.0, x_test / 255.0
num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

In [5]:
# Defining the AlexNet Model (Adjusted for CIFAR-10)

model = Sequential()

# Layer 1
model.add(Conv2D(96, kernel_size=(3,3), strides=(1,1), input_shape=(32,32,3), padding='same'))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))
model.add(BatchNormalization())

# Layer 2
model.add(Conv2D(256, kernel_size=(3,3), strides=(1,1), padding='same'))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))
model.add(BatchNormalization())

# Layer 3
model.add(Conv2D(384, kernel_size=(3,3), strides=(1,1), padding='same'))
model.add(Activation('relu'))

# Layer 4
model.add(Conv2D(384, kernel_size=(3,3), strides=(1,1), padding='same'))
model.add(Activation('relu'))

# Layer 5
model.add(Conv2D(256, kernel_size=(3,3), strides=(1,1), padding='same'))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))

# Flatten
model.add(Flatten())

# Fully Connected Layer 1
model.add(Dense(1024))
model.add(Activation('relu'))
model.add(Dropout(0.5))

# Fully Connected Layer 2
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(10))
model.add(Activation('softmax'))


In [6]:
# Compiling the Model

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
# Training the Model

history = model.fit(x_train, y_train,
                    batch_size=128,
                    epochs=15,
                    validation_split=0.2,
                    verbose=1)

In [ ]:
# Evaluating the Model

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Accuracy: {test_acc:.4f}')

In [ ]:
# Plotting Training & Validation Accuracy

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('AlexNet on CIFAR-10 (GPU)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()